# LLM Workflows (part 2)
This notebook continues from class 01: prompting patterns that expose *reasoning* and *tool use*.

1. [Setup](#setup)
2. [Chain of thought](#chain-of-thought)
3. [ReAct loop (date)](#react-loop-date)
    - [Manual walkthrough](#manual-walkthrough)
    - [Creating the `react_loop`](#creating-react-loop)
4. [Healthbot Agent](#healthbot-agent)
    - [Demo 1 — add a new entry](#demo-1-add-entry)
    - [Demo 2 — answer from history](#demo-2-history)
    - [Wrap it: `healthbot`](#wrap-it-healthbot)
5. [How a modern framework handles this (OpenAI SDK)](#openai-sdk)


<a id="setup"></a>
## Setup
We're creating a small `LLM` helper class for this notebook.  It enables us to define a particular model and provider for each exercise.  To be pythonic, we have two methods that have different signatures: 
- `.send()` returns text 
- `.parse()` returns a Pydantic model via constrained decoding.

To use OpenRouter, you will need to copy [`.env.example`](../.env.example) to `.env` and add your key (`OPENROUTER_API_KEY`).  Do not paste keys into the notebook.


In [ ]:
import os
from dataclasses import dataclass
from datetime import date
from pathlib import Path
from typing import Any, Literal

import requests
from dotenv import load_dotenv
from ollama import chat
from pydantic import BaseModel, Field

load_dotenv()

# --- local ---
local_model = "llama3.2:1b"
# if you have this, you can run a lot of the examples without openrouter
local_model_big = "llama3.1:latest"

# --- OpenRouter: requires login and api key registration ---
# this model is free, but may limit you on tokens
openrouter_model = "nvidia/nemotron-3.5-lightning:free"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# basically saves us from __init__ boilerplate
@dataclass
class LLM:
    """Thin wrapper around a chat model. Set model once; call send or parse."""

    model: str
    # allows us to use one class for either local or openrouter
    provider: Literal["ollama", "openrouter"] = "ollama"
    temperature: float = 0

    def send(self, messages: list[dict]) -> str:
        """Return the model's reply as text."""
        return self._request(messages)

    def parse(self, messages: list[dict], schema: type[BaseModel]) -> BaseModel:
        """Return the model's reply parsed into `schema` (constrained decoding)."""
        content = self._request(messages, schema=schema)
        return schema.model_validate_json(content)

    def _request(self, messages: list[dict], schema: type[BaseModel] | None = None) -> str:
        if self.provider == "ollama":
            response = chat(
                model=self.model,
                messages=messages,
                format=schema.model_json_schema() if schema else None,
                options={"temperature": self.temperature},
            )
            return response.message.content

        return self._request_openrouter(messages, schema=schema)

    def _request_openrouter(self, messages: list[dict], schema: type[BaseModel] | None = None) -> str:
        api_key = os.environ.get("OPENROUTER_API_KEY")
        if not api_key:
            raise RuntimeError("OPENROUTER_API_KEY is not set. Ensure it is set in .env.")

        payload: dict = {
            "model": self.model,
            "messages": messages,
            "temperature": self.temperature,
        }
        if schema is not None:
            payload["response_format"] = {
                "type": "json_schema",
                "json_schema": {
                    "name": schema.__name__,
                    "schema": schema.model_json_schema(),
                },
            }

        r = requests.post(
            OPENROUTER_URL,
            headers={
                "Authorization": f"Bearer {api_key}",
            },
            json=payload,
            timeout=60,
        )
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]


print("local_model:", local_model)
print("openrouter_model:", openrouter_model)


<a id="chain-of-thought"></a>
## Chain of thought
Same question, two prompts: **answer only** vs **step-by-step then answer**.

The text the model writes after "think step by step" is a *generated explanation* — useful for teaching and debugging — not a guaranteed window into the model's internal computation.

Expected answer for this problem: **11** (`5 + 2 × 3`).


In [ ]:
TENNIS_PROBLEM = (
    "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
    "Each can has 3 tennis balls. How many tennis balls does he have now?"
)

# model for this section — swap here to try OpenRouter:
# llm = LLM(model=openrouter_model, provider="openrouter")
llm = LLM(model=local_model)


In [ ]:
# Direct: answer only
direct_prompt = (
    f"{TENNIS_PROBLEM}\n\n"
    "Reply with only the final number. Do not explain."
)
print("PROMPT:\n", direct_prompt)
print("\n---\n")
direct_response = llm.send([{"role": "user", "content": direct_prompt}])
print("RESPONSE:\n", direct_response)


In [ ]:
# Step-by-step: ask for reasoning, then a final answer line
cot_prompt = (
    f"{TENNIS_PROBLEM}\n\n"
    "Think step by step. Show each arithmetic step. "
    "End with a line: Final answer: <number>"
)
print("PROMPT:\n", cot_prompt)
print("\n---\n")
cot_response = llm.send([{"role": "user", "content": cot_prompt}])
print("RESPONSE:\n", cot_response)


<a id="react-loop-date"></a>
## ReAct loop (date)
ReAct = **Re**ason + **Act**: Gives an LLM a structured way to think through a problem and utilize the outputs of its actions (i.e. tool calls).

We're coding this from scratch even though it's a common pattern and one that has mostly been subsumed in modern agents frameworks.  The goal is to show what's going on under the hood.

First we define a "step" schema - a model thinks about what the last input was (from a user or tool), decides whether to take an action and what arguments to use.  We run the tool manually and return the output.

The model eventually decides when it's ready to give a final answer...or we stop it from looping.

Note - we don't use ollama's tool calling here anymore.  It ends up messing with the structured output format.

In [ ]:
import inspect

class ReActStep(BaseModel):
    """One ReAct turn."""
    thought: str = Field(description="Brief reasoning for this step")
    action: str | None = Field(
        default=None,
        description="Tool name to call, or null if giving a final answer",
    )
    action_arguments: dict[str, Any] = Field(
        default_factory=dict,
        description="Keyword arguments for the tool",
    )
    final_answer: str | None = Field(
        default=None,
        description="Answer to the user when done; null if calling a tool",
    )

# from notebook 01
def today(text: str = "") -> str:
    """Returns today's date, use this when you need to get today's date. 
    The input should always be an empty string."""
    return str(date.today())

# useful to lookup tools based on name
date_tool_dict = {"today": today}

def call_tool(step: ReActStep, tool_dict: dict) -> str:
    """Look up the tool by name and call it; errors go back to the model as observations."""
    if step.action not in tool_dict:
        return f"{step.action} not a valid tool"
    try:
        return str(tool_dict[step.action](**step.action_arguments))
    except Exception as e:
        return f"Error calling {step.action}: {e}"

# useful as we build with multiple tools
def react_system(tool_fns: list) -> str:
    """Build a system prompt from tool signatures + docstrings (no private APIs)."""
    tool_lines = []
    for fn in tool_fns:
        args = ", ".join(inspect.signature(fn).parameters)
        desc = (fn.__doc__ or "").strip().split("\n")[0]
        tool_lines.append(f"- {fn.__name__}({args}): {desc}")
    tools = "\n".join(tool_lines)

    return f"""Answer the user's question. You can use tools.

Tools:
{tools}

You work in steps. Each step, reply with JSON containing
thought, action, action_arguments, and final_answer.

To use a tool: set action to the tool name and action_arguments to its arguments. Set final_answer to null.
I will run the tool and reply with "Observation: <result>".

When you know the answer: set action to null and write final_answer.
"""

print("SYSTEM PROMPT:\n", react_system([today]))

# likely this will fail - you need a bigger model, either local_model_big or openrouter_model
# but the failure is interesting to see!
react_llm = LLM(model=local_model)
# react_llm = LLM(model=openrouter_model, provider="openrouter")



<a id="manual-walkthrough"></a>
### Manual walkthrough
One question — "What is today's date?" — stepped cell by cell.


In [ ]:
date_messages = [
    {"role": "system", "content": react_system([today])},
    {"role": "user", "content": "What is today's date?"},
]

step1 = react_llm.parse(date_messages, ReActStep)
print(step1.model_dump_json(indent=2))
date_messages.append({"role": "assistant", "content": step1.model_dump_json()})


In [ ]:
# execute like class 01: look up the function and call with arguments
# no tool calls if we have a final answer
if step1.final_answer:
    print(f"Final answer: {step1.final_answer}")
else:
    # store the output of the tool as an observation (making it a user message for simplicity)
    tool_result = call_tool(step1, date_tool_dict)
    print(f"Observation: {tool_result}")
    date_messages.append({"role": "user", "content": f"Observation: {tool_result}"})

In [ ]:
step2 = react_llm.parse(date_messages, ReActStep)
print(step2.model_dump_json(indent=2))
date_messages.append({"role": "assistant", "content": step2.model_dump_json()})


<a id="creating-react-loop"></a>
### Creating the `react_loop`

Once the pieces are in place (`react_system`, `llm.parse`, tool functions), the end-to-end loop is just composition — same idea as `healthbot_fn` at the end of class 01.


In [ ]:
def react_loop(llm: LLM, question: str, tools: list, max_steps: int = 3) -> str:
    """Compose prompt + parse + tool execution into one ReAct loop."""
    tool_dict = {fn.__name__: fn for fn in tools}
    messages = [
        {"role": "system", "content": react_system(tools)},
        {"role": "user", "content": question},
    ]

    for i in range(max_steps):
        step = llm.parse(messages, ReActStep)
        print(f"=== step {i + 1} ===\n{step.model_dump_json(indent=2)}\n")
        messages.append({"role": "assistant", "content": step.model_dump_json()})

        if step.final_answer:
            return step.final_answer
        else:
            tool_result = call_tool(step, tool_dict)
            print(f"Observation: {tool_result}\n")
            messages.append({"role": "user", "content": f"Observation: {tool_result}"})

    return "(stopped: max_steps)"


print(react_loop(react_llm, "What is today's date?", [today], max_steps=5))


<a id="healthbot-agent"></a>
## Healthbot Agent
If we wanted our HealthBot to be capable of deciding what actions to take, we start getting closer to what the field defines as an "agent".  In this case, we'll use ReAct to have the agent decide what actions to take on behalf of the user, and respond accordingly.

Here's two pathways we'll have in our HealthBot Agent:

1. **add_entry** — Modified workflow; store patient input, symptoms and diagnosis (if provided).
2. **read_entries** — return health records so the model can answer a history question.

Technically, there's a third pathway, which is to refuse to respond, in case none of the tools are appropriate.


In [ ]:
# Schemas reused from class 01
class Diagnosis(BaseModel):
    diagnosis: str = Field(description="The likely diagnosis")
    confidence: str = Field(
        description="How confident we are in the diagnosis"
    )

class Record(BaseModel):
    patient_input: str
    symptoms: list[str]
    search_query: str | None = None
    search_results: str | None = None
    diagnosis: Diagnosis | None = None

RECORDS_PATH = Path("records.jsonl")


In [ ]:
# Seed some records, overwrite the existing to keep reproducible
seed = [
    Record(
        patient_input="I've been coughing and sneezing for two days.",
        symptoms=["coughing", "sneezing"],
        diagnosis=Diagnosis(diagnosis="Common cold", confidence="medium"),
    ),
    Record(
        patient_input="Pounding headache behind my eyes, light hurts, nauseous.",
        symptoms=["headache", "light sensitivity", "nausea"],
        diagnosis=Diagnosis(diagnosis="Migraine", confidence="medium"),
    ),
]

with RECORDS_PATH.open("w") as f:
    for rec in seed:
        f.write(rec.model_dump_json() + "\n")

print(f"Seeded {len(seed)} records -> {RECORDS_PATH.resolve()}")
print(RECORDS_PATH.read_text())


In [ ]:
def add_entry(
    patient_input: str,
    symptoms: list[str],
    diagnosis: str | None = None,
    confidence: str | None = None,
) -> str:
    """Save a new health record .

    Args:
        patient_input: What the patient said.
        symptoms: Short symptom strings.
        diagnosis: Likely diagnosis label.
        confidence: One of low, medium, or high.
    """
    record = Record(
        patient_input=patient_input,
        symptoms=symptoms.split(",") if isinstance(symptoms, str) else symptoms,
        search_query=None,
        search_results=None,
        diagnosis=Diagnosis(diagnosis=diagnosis, confidence=confidence) if diagnosis else None,
    )
    with RECORDS_PATH.open("a") as f:
        f.write(record.model_dump_json() + "\n")
    return f"Saved record: {record.model_dump_json()}"


def read_entries(text: str = "") -> str:
    """Return ALL prior records from records.jsonl as text.

    Args:
        text: Unused; pass an empty string.
    """
    if not RECORDS_PATH.exists():
        return "No records yet."
    return RECORDS_PATH.read_text().strip() or "No records yet."


health_tools = [add_entry, read_entries]
health_tool_dict = {fn.__name__: fn for fn in health_tools}


<a id="demo-1-add-entry"></a>
### Demo 1 — add a new entry
Looking at the new entry flow.


In [ ]:
add_question = (
    "Please record this visit: I have a sore throat and mild fever. "
    "Symptoms: sore throat, fever."
)

add_result = react_loop(react_llm, add_question, health_tools)
print("Final:\n", add_result)
print("\n--- records.jsonl ---\n")
print(RECORDS_PATH.read_text())


<a id="demo-2-history"></a>
### Demo 2 — answer from history
Let's look at an example with just the history flow.

In [ ]:
history_question = (
    "Looking at all prior visits, what symptoms have we recorded so far? "
    "List them briefly."
)

history_result = react_loop(react_llm, history_question, health_tools)
print("Final:\n", history_result)


<a id="wrap-it-healthbot"></a>
### Wrap it: `healthbot`

Same as class 01: once the loop exists, the app is just a thin composition over the tools.


In [ ]:
def healthbot(question: str) -> str:
    return react_loop(react_llm, question, health_tools)


print(healthbot(
    "Summarize the diagnoses we have on file in one short sentence."
))


It's also worth noting that the agent can also choose not to use any of the tools and just answer directly.

In [ ]:
print(healthbot("Hello!"))

<a id="openai-sdk"></a>
## How a modern framework handles this (OpenAI SDK)

Of course we probably don't want to implement ReAct from scratch every time we want to build an agent.  Here's how to do it in OpenAI's SDK.  

**Note**: This will not work with the base environment, you'll need to install `pixi install -e extras` first.

In [ ]:
from agents import (
    Agent,
    Runner,
    function_tool,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
)

tool_add_entry = function_tool(add_entry)

tool_read_entries = function_tool(read_entries)

client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

model = OpenAIChatCompletionsModel(
    model=openrouter_model,
    openai_client=client,
)

agent = Agent(
    name="Healthbot",
    instructions="You are a brief health journal assistant.",
    model=model,
    tools=[tool_add_entry, tool_read_entries],
)

question = "Please record this visit: I have a sore throat and mild fever. "

result = await Runner.run(agent, question)
print(result.final_output)